# Review-to-Revenue: Agent Build, Demonstration, and Evaluation

This notebook defines and tests the restaurant comparison agent.

It assumes `01_data_pipeline.ipynb` has already created the Unity Catalog tables and Databricks Vector Search index.

**Agent stack**: 
- LLM: Meta Llama 3.3 70B Instruct (Databricks Foundation Model API)
- Embeddings: GTE Large (En) for vector search
- Framework: LangGraph for tool routing

## Table of Contents


### AI Engineering
7. **Create Agent Tools** - Build the three core agent capabilities
   - 7.1 Tool 1 – Business Lookup
   - 7.2 Tool 2 – Competitor Comparison
   - 7.3 Initialize Vector Search Client
   - 7.4 Helper Function to Parse Vector Search Results
   - 7.5 Tool 3 – Theme Retrieval
   - 7.6 Validate All Three Tools

8. **Agent Creation & Testing** - Build ReAct agent with both LLMs
   - 8.1 Configure LLM Endpoint
   - 8.2 Define Tool Metadata
   - 8.3 Define Tool Specifications
   - 8.4 Register Tools
   - 8.5 Create System Prompt
   - 8.6 Create Tool-Calling Agent Class with MLflow Tracing
   - 8.7 Create Agents for Both LLMs
   - 8.8 Compare Both LLMs on the Same Prompts

9. **Evaluation Agent with LLM Judges** - Automated quality evaluation
   - 9.1 Set MLflow Experiment
   - 9.2 Create Evaluation Questions
   - 9.3 Convert Questions to MLflow Evaluation Format
   - 9.4 Create Prediction Function
   - 9.5 Create T/F LLM Judge
   - 9.6 Run LLM Judge Evaluation

### Product Management
10. **Human Review Layer** - Manual validation of LLM judge scores
   - 10.1 Prepare Evaluation Materials
   - 10.2 Team Review for Responses

11. **ROI Analysis - Model Comparison** - Cost-benefit analysis for model selection
   - 11.1 Extract Token Usage from MLflow Traces
   - 11.2 Define Pricing and Calculate Costs
   - 11.3 Aggregate Quality Scores
   - 11.4 Calculate Business Value
   - 11.5 Calculate Final ROI
   - 11.6 Generate Comparison Tables
   - 11.7 Sensitivity Analysis
   - 11.8 Final Recommendation

---

In [0]:
# Install required packages for the agent notebook
%pip install databricks-vectorsearch openai

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F

# Unity Catalog namespace created by the data pipeline notebook
CATALOG = "main"
SCHEMA = "aai510_final_agent"

# Source tables created by 01_data_pipeline.ipynb
RESTAURANT_METRICS_TABLE = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
RESTAURANTS_TABLE = f"{CATALOG}.{SCHEMA}.restaurants"
REVIEWS_WITH_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"
REVIEWS_FOR_VECTOR_TABLE = f"{CATALOG}.{SCHEMA}.reviews_for_vector_search"

# Existing Vector Search assets created by 01_data_pipeline.ipynb
VECTOR_SEARCH_ENDPOINT_NAME = "restaurant_reviews_vs_endpoint"
VECTOR_SEARCH_INDEX_NAME = f"{CATALOG}.{SCHEMA}.restaurant_review_text_index"
EMBEDDING_MODEL_ENDPOINT_NAME = "databricks-gte-large-en"

print("=" * 60)
print("Configured existing data assets for the agent notebook")
print("=" * 60)
print(f"Restaurant metrics table: {RESTAURANT_METRICS_TABLE}")
print(f"Restaurants table: {RESTAURANTS_TABLE}")
print(f"Review text table: {REVIEWS_WITH_TEXT_TABLE}")
print(f"Vector search source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Vector Search endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
print(f"Vector Search index: {VECTOR_SEARCH_INDEX_NAME}")


Configured existing data assets for the agent notebook
Restaurant metrics table: main.aai510_final_agent.restaurant_metrics
Restaurants table: main.aai510_final_agent.restaurants
Review text table: main.aai510_final_agent.reviews_with_text
Vector search source table: main.aai510_final_agent.reviews_for_vector_search
Vector Search endpoint: restaurant_reviews_vs_endpoint
Vector Search index: main.aai510_final_agent.restaurant_review_text_index


---
##7. Create Agent Tools

Convert the restaurant data pipeline into reusable tools for the agent. These tools allow the agent to answer both structured questions about ratings and review counts and qualitative questions about customer experiences.

1. **Business Lookup**: Retrieve rating, review count, rating distribution, and restaurant metadata.
2. **Competitor Comparison**: Compare restaurants by rating, review volume, and rating distribution.
3. **Theme Retrieval**: Use vector search over review text to retrieve reviews related to themes like service, food quality, ambience, price, wait time, or complaints.

### 7.1. Tool 1 - Business Lookup

Objective: Retrieve metrics for a specific restaurant

In [0]:
def business_lookup(restaurant_name: str, limit: int = 5):
    """
    Retrieve rating, review count, rating distribution, and restaurant metadata
    for a specific restaurant.
    """
    
    restaurant_metrics_df = spark.table(RESTAURANT_METRICS_TABLE)
    
    results_df = (
        restaurant_metrics_df
        .filter(F.lower(F.col("business_name")).contains(restaurant_name.lower()))
        .select(
            "gmap_id",
            "business_name",
            "address",
            "avg_rating",
            "review_count",
            "rating_1_count",
            "rating_2_count",
            "rating_3_count",
            "rating_4_count",
            "rating_5_count"
        )
        .orderBy(F.desc("review_count"))
        .limit(limit)
    )
    
    results = results_df.toPandas().to_dict(orient="records")
    
    if len(results) == 0:
        return {
            "status": "not_found",
            "message": f"No restaurant found matching '{restaurant_name}'."
        }
    
    return {
        "status": "success",
        "results": results
    }

### 7.2. Tool 2 - Competitor Comparison

Objective: Compare restaurant performance to another restaurant

In [0]:
def competitor_comparison(restaurant_1: str, restaurant_2: str, limit_per_name: int = 3):
    """
    Compare two specific restaurants by rating, review volume, and rating distribution.
    The user provides both restaurant names.
    """
    
    restaurant_metrics_df = spark.table(RESTAURANT_METRICS_TABLE)
    
    restaurant_names = [restaurant_1, restaurant_2]
    all_results = []
    
    for restaurant_name in restaurant_names:
        matches_df = (
            restaurant_metrics_df
            .filter(F.lower(F.col("business_name")).contains(restaurant_name.lower()))
            .select(
                "gmap_id",
                "business_name",
                "address",
                "avg_rating",
                "review_count",
                "rating_1_count",
                "rating_2_count",
                "rating_3_count",
                "rating_4_count",
                "rating_5_count"
            )
            .orderBy(F.desc("review_count"))
            .limit(limit_per_name)
        )
        
        matches = matches_df.toPandas().to_dict(orient="records")
        
        for match in matches:
            match["search_name"] = restaurant_name
        
        all_results.extend(matches)
    
    if len(all_results) == 0:
        return {
            "status": "not_found",
            "message": f"No matching restaurants found for '{restaurant_1}' or '{restaurant_2}'."
        }
    
    return {
        "status": "success",
        "restaurant_1": restaurant_1,
        "restaurant_2": restaurant_2,
        "results": all_results
    }

### 7.3. Initialize Vector Search Client

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

print("Vector Search client initialized.")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Vector Search client initialized.


### 7.4. Helper Function to Parse Vector Search Results

In [0]:
def parse_vector_search_results(results):
    """
    Convert Databricks Vector Search response into a list of dictionaries.
    """
    
    columns = [col["name"] for col in results["manifest"]["columns"]]
    rows = results["result"]["data_array"]
    
    parsed_results = []
    
    for row in rows:
        parsed_results.append(dict(zip(columns, row)))
    
    return parsed_results

### 7.5. Tool 3 - Theme Retrieval

Objective: Find relevant reviews by meaning, not just keywords.

In [0]:
def theme_retrieval(restaurant_name: str, theme_query: str, num_results: int = 10):
    """
    Retrieve reviews related to a user-provided theme using Databricks Vector Search.
    This uses semantic similarity search over review text and does not rely on
    hard-coded keywords.
    """
    
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    # Pull more than needed because restaurant filtering happens after retrieval.
    raw_num_results = max(num_results * 5, 25)
    
    results = review_index.similarity_search(
        query_text=theme_query,
        columns=[
            "review_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ],
        num_results=raw_num_results
    )
    
    parsed_results = parse_vector_search_results(results)
    
    filtered_results = [
        row for row in parsed_results
        if restaurant_name.lower() in row.get("business_name", "").lower()
    ]
    
    filtered_results = filtered_results[:num_results]
    
    if len(filtered_results) == 0:
        return {
            "status": "not_found",
            "message": (
                f"No vector search results found for restaurant '{restaurant_name}' "
                f"and theme query '{theme_query}'."
            ),
            "restaurant_name": restaurant_name,
            "theme_query": theme_query,
            "results": []
        }
    
    return {
        "status": "success",
        "restaurant_name": restaurant_name,
        "theme_query": theme_query,
        "results": filtered_results
    }

### 7.6 Validate All Three Tools

In [0]:
print("=" * 60)
print("Validating agent tools...")
print("=" * 60)

print("\nTool 1: Business Lookup")
business_lookup_result = business_lookup("Cesarina")

if business_lookup_result["status"] == "success":
    display(business_lookup_result["results"])
else:
    print(business_lookup_result["message"])


print("\nTool 2: Competitor Comparison")
comparison_result = competitor_comparison("Cesarina", "Parma Cucina Italiana")

if comparison_result["status"] == "success":
    display(comparison_result["results"])
else:
    print(comparison_result["message"])


print("\nTool 3: Theme Retrieval")
try:
    theme_result = theme_retrieval(
        "Cesarina",
        "reviews about pasta quality and authentic Italian food",
        num_results=5
    )
    
    if theme_result["status"] == "success":
        display(theme_result["results"])
    else:
        print(theme_result["message"])

except Exception as e:
    print("Theme Retrieval failed.")
    print(e)

Validating agent tools...

Tool 1: Business Lookup


address,avg_rating,business_name,gmap_id,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count,review_count
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",4.840304182509506,Cesarina,0x80deaa30b08c1367:0x807034f5191ec879,2,2,11,48,463,526



Tool 2: Competitor Comparison


address,avg_rating,business_name,gmap_id,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count,review_count,search_name
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",4.840304182509506,Cesarina,0x80deaa30b08c1367:0x807034f5191ec879,2,2,11,48,463,526,Cesarina
"Parma Cucina Italiana, 3850 Fifth Ave, San Diego, CA 92103",4.670289855072464,Parma Cucina Italiana,0x80d954da94044cfb:0x9830f72553b8350c,7,7,24,85,429,552,Parma Cucina Italiana



Tool 3: Theme Retrieval
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


address,business_name,rating,review_id,score,text,time
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,c8bcf70b7a4bf731ac20a3230af624139cd169b14f6b4e63ec1d93b93d1cac21,0.6831242105483257,Amazing fresh pasta,1.578022539062E12
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,af652a30074e57e2e35b79a5286239ace603176d61cab25dffcb0413c49b565f,0.6708889848396832,"Excellent homemade pasta, highly recommended",1.560981598778E12
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,c32b474c7073b02a6db1590dae2428250ca30615cfa91453e6fb87a5fe9d11c1,0.666536010261426,"Seriously good, real home made pastas, must try gnocchi, pappardelle, rigatoni. Pick your own sauce and of course owners are so acomodating. Great experience thank you",1.559279660709E12
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,759b96259509e0be622c3e5ae4ebedebba8bceb5ce400a4de1538620ed05880c,0.6656673520125519,What a wonderful discovery of excellent of fresh italian pasta with excellent service! Yum!,1.564346874129E12
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,dde684d7e241ce7103b7a170f3341734a0bad48ddd8c4caa38d554db23a93c67,0.6565337575663645,"Outstanding Italian restaurant with great ambiance. The food in general, and pasta in particular, deserve six stars!!! Everything is made fresh and taste very authentic, like as if it was in the heart of Italy. Highly recommended.",1.550029505853E12


---
## 8. Create Restaurant Comparison Agent

This section connects the reusable restaurant tools to a tool-calling agent. The agent uses an LLM to interpret the user’s question, choose the appropriate tool, call the tool, and synthesize a final answer using the tool results.

### 8.1. Configure LLM Endpoint

Baseline: Llama 3.3 70B
Comparirson: GPT OSS 20b (cheaper model)

In [0]:
# LLM endpoints used for the restaurant comparison agent

LLAMA_3_3_70B_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
GPT_OSS_20B_ENDPOINT = "databricks-gpt-oss-20b"

AVAILABLE_LLMS = [
    LLAMA_3_3_70B_ENDPOINT,
    GPT_OSS_20B_ENDPOINT
]

# Choose the default LLM for the first agent run
CHOSEN_LLM_ENDPOINT = LLAMA_3_3_70B_ENDPOINT

print("=" * 60)
print("Configuring LLM endpoints...")
print("=" * 60)

print("Available LLM endpoints:")
for endpoint in AVAILABLE_LLMS:
    print(f"- {endpoint}")

print(f"\nDefault LLM endpoint: {CHOSEN_LLM_ENDPOINT}")

Configuring LLM endpoints...
Available LLM endpoints:
- databricks-meta-llama-3-3-70b-instruct
- databricks-gpt-oss-20b

Default LLM endpoint: databricks-meta-llama-3-3-70b-instruct


### 8.2. Define Tool Metadata

In [0]:
import json
from dataclasses import dataclass
from typing import Callable

@dataclass
class ToolInfo:
    name: str
    spec: dict
    exec_fn: Callable

### 8.3. Define Tool Specifications

In [0]:
# Tool specifications in OpenAI-compatible function-calling format

business_lookup_spec = {
    "type": "function",
    "function": {
        "name": "business_lookup",
        "description": (
            "Retrieve rating, review count, rating distribution, and restaurant metadata "
            "for a specific restaurant."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_name": {
                    "type": "string",
                    "description": "Name or partial name of the restaurant to look up."
                },
                "limit": {
                    "type": "integer",
                    "description": "Maximum number of matching restaurants to return."
                }
            },
            "required": ["restaurant_name"]
        }
    }
}


competitor_comparison_spec = {
    "type": "function",
    "function": {
        "name": "competitor_comparison",
        "description": (
            "Compare two specific restaurants by rating, review volume, and rating distribution."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_1": {
                    "type": "string",
                    "description": "Name or partial name of the first restaurant."
                },
                "restaurant_2": {
                    "type": "string",
                    "description": "Name or partial name of the second restaurant."
                },
                "limit_per_name": {
                    "type": "integer",
                    "description": "Maximum number of matches to return per restaurant name."
                }
            },
            "required": ["restaurant_1", "restaurant_2"]
        }
    }
}


theme_retrieval_spec = {
    "type": "function",
    "function": {
        "name": "theme_retrieval",
        "description": (
            "Retrieve reviews related to a user-provided theme using semantic Vector Search "
            "over restaurant review text. Use this for qualitative questions about customer experiences, "
            "including service, food quality, ambiance, price, wait time, complaints, and special occasions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_name": {
                    "type": "string",
                    "description": "Name or partial name of the restaurant."
                },
                "theme_query": {
                    "type": "string",
                    "description": (
                        "Natural-language theme to search for in review text. "
                        "For example: 'service and staff friendliness', "
                        "'pasta quality and authentic Italian food', or "
                        "'ambiance for date night'."
                    )
                },
                "num_results": {
                    "type": "integer",
                    "description": "Number of relevant review excerpts to retrieve."
                }
            },
            "required": ["restaurant_name", "theme_query"]
        }
    }
}

### 8.4. Register Tools

In [0]:
TOOL_INFOS = [
    ToolInfo(
        name="business_lookup",
        spec=business_lookup_spec,
        exec_fn=business_lookup
    ),
    ToolInfo(
        name="competitor_comparison",
        spec=competitor_comparison_spec,
        exec_fn=competitor_comparison
    ),
    ToolInfo(
        name="theme_retrieval",
        spec=theme_retrieval_spec,
        exec_fn=theme_retrieval
    )
]

print("=" * 60)
print("Registering agent tools...")
print("=" * 60)

print(f"Registered {len(TOOL_INFOS)} tools:")
for tool in TOOL_INFOS:
    print(f"- {tool.name}")

Registering agent tools...
Registered 3 tools:
- business_lookup
- competitor_comparison
- theme_retrieval


### 8.5. Create System Prompt

In [0]:
SYSTEM_PROMPT = """
You are a restaurant comparison assistant focused on San Diego Italian restaurants.

Your scope:
- Answer questions about San Diego Italian restaurants using the available restaurant metrics and review-text tools.
- Help users compare restaurants, understand ratings, review counts, rating distributions, and customer review themes.
- Provide recommendations only when they can be grounded in the available tool outputs.

Out-of-scope requests:
- Do not answer questions unrelated to San Diego Italian restaurants.
- Do not provide general homework help, coding help, medical advice, legal advice, financial advice, travel planning, politics, sports, entertainment, or unrelated factual information.
- Do not invent restaurant data, restaurant names, metrics, ratings, review counts, review text, or business recommendations.

If the user asks an out-of-scope question, politely reject it and briefly explain that you can only help with San Diego Italian restaurant comparison questions using the available data.

You have access to three tools:
1. business_lookup: retrieves restaurant-level metrics such as average rating, review count, and rating distribution.
2. competitor_comparison: compares two restaurants using rating, review volume, and rating distribution.
3. theme_retrieval: retrieves semantically relevant review excerpts for qualitative customer-experience themes.

Tool-use rules:
- For structured questions about one restaurant, use business_lookup.
- For structured comparison questions about two restaurants, use competitor_comparison.
- For qualitative questions about customer experiences, use theme_retrieval.
- For broad comparison, recommendation, or “what does one restaurant do better” questions, use both competitor_comparison and theme_retrieval. The structured comparison shows rating and review-count differences, while theme_retrieval provides specific evidence about what customers mention, such as service, food quality, ambiance, authenticity, value, wait time, and special occasions.
- If a tool returns no results, say that the data was not found.

Use the restaurant names provided by the user. Do not assume the user only cares about a fixed pair of restaurants.
Be concise, but include enough evidence from tool outputs to justify the answer.
"""

### 8.6. Create Tool-Calling Agent Class with MLflow Tracing

In [0]:
import mlflow
from databricks.sdk import WorkspaceClient

# Enable MLflow tracing for OpenAI-compatible chat calls
mlflow.autolog()

print("MLflow OpenAI autologging enabled.")

2026/06/22 17:20:49 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for pyspark: MLflow Spark dataset autologging is not supported on Databricks shared clusters or Databricks serverless clusters.
2026/06/22 17:20:49 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for pyspark.ml: [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `sparkContext` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.


MLflow OpenAI autologging enabled.


In [0]:
class RestaurantToolCallingAgent:
    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        self.llm_endpoint = llm_endpoint
        self.tools = {tool.name: tool for tool in tools}
        self.workspace_client = WorkspaceClient()
        self.client = self.workspace_client.serving_endpoints.get_open_ai_client()
    
    def get_tool_specs(self):
        return [tool.spec for tool in self.tools.values()]
    
    def execute_tool(self, tool_name: str, arguments: dict):
        if tool_name not in self.tools:
            return {
                "status": "error",
                "message": f"Unknown tool: {tool_name}"
            }
        
        tool = self.tools[tool_name]
        return tool.exec_fn(**arguments)
    
    def predict(self, user_question: str, max_tool_rounds: int = 5, verbose: bool = True):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question}
        ]
        
        for _ in range(max_tool_rounds):
            response = self.client.chat.completions.create(
                model=self.llm_endpoint,
                messages=messages,
                tools=self.get_tool_specs(),
                tool_choice="auto"
            )
            
            assistant_message = response.choices[0].message
            messages.append(assistant_message.model_dump(exclude_none=True))
            
            # If the model does not call a tool, return the final response
            if not assistant_message.tool_calls:
                return assistant_message.content
            
            # Execute tool calls
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments or "{}")
                
                if verbose:
                    print(f"Calling tool: {tool_name}")
                    print(f"Arguments: {arguments}")
                
                tool_result = self.execute_tool(tool_name, arguments)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result, default=str)
                })
        
        return "The agent reached the maximum number of tool calls before producing a final answer."

### 8.7. Create Agents for Both LLMs

In [0]:
AGENT_LLAMA_3_3_70B = RestaurantToolCallingAgent(
    llm_endpoint=LLAMA_3_3_70B_ENDPOINT,
    tools=TOOL_INFOS
)

AGENT_GPT_OSS_20B = RestaurantToolCallingAgent(
    llm_endpoint=GPT_OSS_20B_ENDPOINT,
    tools=TOOL_INFOS
)

# Default agent
AGENT = AGENT_LLAMA_3_3_70B

print("=" * 60)
print("Restaurant comparison agents created.")
print("=" * 60)

print(f"Default agent: {LLAMA_3_3_70B_ENDPOINT}")
print(f"Comparison agent: {GPT_OSS_20B_ENDPOINT}")

Restaurant comparison agents created.
Default agent: databricks-meta-llama-3-3-70b-instruct
Comparison agent: databricks-gpt-oss-20b


### 8.8. Compare Both LLMs on the Same Prompts

#### Prompt 1

In [0]:
test_question = (
    "How many reviews does Parma Cucina Italiana have, and what is its average rating?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
Calling tool: business_lookup
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'limit': 1}
Parma Cucina Italiana has an average rating of 4.67 and 552 reviews.

GPT OSS 20B


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ting. Use the tool.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Calling tool: business_lookup
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'limit': 1}
Parma Cucina Italiana has **552 reviews** and an **average rating of 4.67** (rounded to two decimal places).


#### Prompt 2

In [0]:
test_question = (
    "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
Calling tool: competitor_comparison
Arguments: {'restaurant_1': 'Parma Cucina Italiana', 'restaurant_2': 'Cesarina', 'limit_per_name': 1}
Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Cesarina', 'theme_query': 'food quality and authentic Italian food', 'num_results': 5}
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Based on the comparison of Parma Cucina Italiana and Cesarina, Cesarina appears to have a slightly higher average rating (4.84 vs 4.67) and a higher proportion of 5-star reviews. The theme retrieval results for Cesarina also suggest that customers praise the restaurant for its authentic Italian food, great service, and high-quality dishes. Therefore, it seems that Cesarina does better in terms of food quality and authenticity.

GPT OSS 20B


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...er. Let\'s do that.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Calling tool: competitor_comparison
Arguments: {'restaurant_1': 'Parma Cucina Italiana', 'restaurant_2': 'Cesarina', 'limit_per_name': 1}


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...city". Let\'s call.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Cesarina', 'theme_query': 'food quality and authenticity', 'num_results': 3}
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Cesarina', 'theme_query': 'service and staff friendliness', 'num_results': 3}


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...rieval for service.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'We have enough. Now produce answer.\n\n'}]}, {'type': 'text', 'text': '**Structured comparison**\n\n| Metric | Parma\u202fCucina Italiana | Cesarina |\n|--------|-----------------------|----------|\n| Avg. rating | **4.67** | **4.84** |\n| Review count | **552** | **526** |\n| 5‑star reviews | **429** | **463** |\n| 4‑star reviews | **85** | **48** |\n| 3‑star reviews | **24** | **11** |\n| 2‑star reviews | **7** | **2** |\n| 1‑star reviews | **7** | **2** |\n\n*Key takeaways:*  \n- Cesarina has a higher average rating (∼0.17 points higher) and a higher proportion of 5‑star reviews.  \n- Both restaurants have similar total review volumes (~530–550).  \n- The rating distribution shows fewer lower‑star reviews for

/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...e the clearer choice.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


#### Prompt 3

In [0]:
test_question = (
    "What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'theme_query': 'pasta quality and Italian authenticity', 'num_results': 5}
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Based on the review excerpts, customers are saying that Parma Cucina Italiana has "authentic Italian" cuisine, with one reviewer specifically mentioning that it is "Very good and authentic Italian". Another reviewer praised the restaurant, saying it has "Great Italian food". The reviews suggest that the pasta quality and Italian authenticity at Parma Cucina Italiana are well-regarded by customers, with ratings of 4.0 and 5.0 out of 5.0.

GPT OSS 20B


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...st theme_retrieval.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'theme_query': 'pasta quality and Italian authenticity', 'num_results': 5}
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Customers consistently praise Parma Cucina Italiana for the quality and authenticity of its pasta dishes. Here are a few excerpts that capture that sentiment:

| Review (rating) | What the reviewer highlighted |
|-----------------|------------------------------|
| “Authentic Italian” (5 ★) | Short, but the reviewer’s score reflects a positive experience with the pasta’s flavor and authenticity. |
| “Very good and authentic Italian” (4 ★) | Notes both the high quality and the genuine Italian cooking style. |
| “Great Italian food” (5 ★) | Emphasizes overall authenticity and taste, extending to the pasta and broader menu. |

#### Prompt 4

In [0]:
test_question = (
    "What is the best recipe for homemade lasagna?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
I can only help with San Diego Italian restaurant comparison questions using the available data. I'm not able to provide general recipes or cooking advice. If you have a question about a specific San Diego Italian restaurant, such as their ratings, review counts, or comparison to another restaurant, I'd be happy to try and assist you.

GPT OSS 20B
[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'User asks about recipe for homemade lasagna. This is out-of-scope: The assistant can only answer questions about San Diego Italian restaurants and comparison using tools. Must refuse politely and explain scope limitation.'}]}, {'type': 'text', 'text': 'I’m sorry, but I can only help with questions about San\u202fDiego Italian restaurants and their reviews. If you’d like to compare restaurants, learn about their ratings, or get insights from customer feedback, just let me know!'}]


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ck, just let me know!'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


#### Prompt 5

In [0]:
test_question = (
    "Can you summarize reviews for a dentist in New York?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
I can only help with San Diego Italian restaurant comparison questions using the available data. Unfortunately, I don't have information about dentists in New York, so I won't be able to summarize reviews for you. If you have any questions about San Diego Italian restaurants, I'd be happy to try and assist you.

GPT OSS 20B
[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'The user: "Can you summarize reviews for a dentist in New York?" The user\'s question is about a dentist in New York, which is not San Diego Italian restaurants. According to the instructions, we must refuse out-of-scope request. Provide brief apology and explain that we can only help with San Diego Italian restaurant comparison questions.'}]}, {'type': 'text', 'text': 'I’m sorry, but I can’t help with that. I’m only able to provide information and comparisons for San\xa0Diego Italian restaurants. If you have any questions about those, feel free to ask!'}]


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...se, feel free to ask!'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


## 9. Evaluation Agent with LLM Judges

This section evaluates the restaurant comparison agent using LLM-as-a-judge scoring. The evaluation checks whether the agent answers the user’s question, stays grounded in tool outputs, uses the appropriate tools, and gracefully rejects out-of-scope questions. The evaluation is run through MLflow so that traces and judge scores can be reviewed in the Databricks Experiments UI.

### 9.1. Set MLflow Experiment

In [0]:
import mlflow

# Set MLflow experiment for final project traces and evaluation
current_user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_NAME = f"/Users/{current_user}/aai510_restaurant_agent_final"

mlflow.set_experiment(EXPERIMENT_NAME)

print("=" * 60)
print("MLflow experiment configured...")
print("=" * 60)
print(f"Experiment: {EXPERIMENT_NAME}")

MLflow experiment configured...
Experiment: /Users/erika3gallegos@gmail.com/aai510_restaurant_agent_final


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


### 9.2. Create Evaluation Questions

In [0]:
# Evaluation questions for LLM judge
# These are the same types of prompts tested in Section 7

EVALUATION_QUESTIONS = [
    {
        "question": "How many reviews does Parma Cucina Italiana have, and what is its average rating?",
        "expected_tools": ["business_lookup"]
    },
    {
        "question": "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?",
        "expected_tools": ["competitor_comparison"]
    },
    {
        "question": (
            "What are customers saying about the service and staff friendliness at Parma Cucina Italiana?"
        ),
        "expected_tools": ["theme_retrieval"]
    },
    {
        "question": "What is the best recipe for homemade lasagna?",
        "expected_tools": []
    },
    {
        "question": "Can you summarize reviews for a dentist in New York?",
        "expected_tools": []
    }
]

print("=" * 60)
print("Evaluation questions created...")
print("=" * 60)
print(f"Total evaluation questions: {len(EVALUATION_QUESTIONS)}")

display(EVALUATION_QUESTIONS)

Evaluation questions created...
Total evaluation questions: 5


expected_tools,question
List(business_lookup),"How many reviews does Parma Cucina Italiana have, and what is its average rating?"
List(competitor_comparison),Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?
List(theme_retrieval),What are customers saying about the service and staff friendliness at Parma Cucina Italiana?
List(),What is the best recipe for homemade lasagna?
List(),Can you summarize reviews for a dentist in New York?


### 9.3. Convert Questions to MLflow Evaluation Format

In [0]:
# Convert Section 8 evaluation questions into MLflow GenAI evaluation format
# MLflow requires expectations to be a dictionary

eval_data = []

for item in EVALUATION_QUESTIONS:
    expected_tools_text = ", ".join(item.get("expected_tools", []))

    if expected_tools_text == "":
        expected_tools_text = "none; the agent should reject the request if it is out of scope"

    eval_data.append(
        {
            "inputs": {
                "question": item["question"]
            },
            "expectations": {
                "expected_tools": expected_tools_text,
                "expected_behavior": (
                    "The agent should answer the question using the appropriate restaurant analysis tools, "
                    "stay grounded in retrieved reviews, business metadata, or tool outputs, "
                    "provide specific and useful business insight, and reject the request gracefully if it is out of scope."
                )
            }
        }
    )

print("=" * 60)
print("Evaluation dataset created...")
print("=" * 60)
print(f"Total evaluation records: {len(eval_data)}")

# Do not use display(eval_data) here because Databricks may not infer nested schemas correctly
for row in eval_data:
    print("\nQuestion:")
    print(row["inputs"]["question"])
    print("Expected tools:")
    print(row["expectations"]["expected_tools"])
    print("Expected behavior:")
    print(row["expectations"]["expected_behavior"])

Evaluation dataset created...
Total evaluation records: 5

Question:
How many reviews does Parma Cucina Italiana have, and what is its average rating?
Expected tools:
business_lookup
Expected behavior:
The agent should answer the question using the appropriate restaurant analysis tools, stay grounded in retrieved reviews, business metadata, or tool outputs, provide specific and useful business insight, and reject the request gracefully if it is out of scope.

Question:
Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?
Expected tools:
competitor_comparison
Expected behavior:
The agent should answer the question using the appropriate restaurant analysis tools, stay grounded in retrieved reviews, business metadata, or tool outputs, provide specific and useful business insight, and reject the request gracefully if it is out of scope.

Question:
What are customers saying about the service and staff friendliness at Parma Cucina Italiana?
Expected tools:
them

### 9.4. Create Prediction Function

In [0]:
def predict_fn(question: str) -> str:
    """
    Run the default restaurant agent on a single evaluation question.
    The default agent uses Llama 3.3 70B.
    """
    
    response = AGENT.predict(
        question,
        verbose=False
    )
    
    return response

print("Prediction function created.")

Prediction function created.


In [0]:
## quick test
predict_fn("How many reviews does Parma Cucina Italiana have, and what is its average rating?")

'Parma Cucina Italiana has an average rating of 4.67 and 552 reviews.'

### 9.5. Create T/F LLM Judge

In [0]:
from mlflow.genai.judges import make_judge

restaurant_owner_judge = make_judge(
    name="restaurant_owner_judge",
    instructions="""
You are evaluating a restaurant-review analysis agent for a final project.

The agent's purpose is to help a small restaurant owner understand San Diego Italian restaurant performance using:
- business metadata,
- rating metrics,
- rating distributions,
- retrieved review evidence,
- competitor comparison outputs,
- and tool outputs from the agent.

User input:
{{ inputs }}

Agent response:
{{ outputs }}

Expected behavior:
{{ expectations }}

Trace:
{{ trace }}

You must evaluate the agent response using the criteria below.

Return TRUE only if all required criteria are satisfied.
Return FALSE if any required criterion is not satisfied.

Criteria:

1. Groundedness:
Is the response grounded in the retrieved reviews, business metadata, or tool outputs?
True if the response does not invent unsupported facts or make claims that cannot be traced to the available evidence.

2. Answer relevance:
Does the response answer the user’s actual question?
True if the response directly addresses the user’s request rather than giving a generic or unrelated answer.

3. Restaurant-owner usefulness:
Is the response useful for a restaurant owner?
True if the response provides practical insight, decision support, or a clear next step.

4. Specificity:
Is the response specific enough to be meaningful?
True if the response includes concrete themes, complaints, strengths, comparisons, or recommendations rather than vague statements.

5. Appropriate tool use:
Did the agent use the appropriate tool or tool sequence?
True if the tool use matches the query type, such as lookup for metadata, retrieval for review evidence, or competitor comparison for comparison questions.

6. Out-of-scope handling:
If the input was irrelevant or out of scope, did the agent reject it gracefully?
True if the agent politely refused or redirected the user to an appropriate restaurant-review analysis task.
For in-scope questions, treat this criterion as satisfied.

7. Clarity:
Is the response clear and understandable for a small business owner?
True if the response is organized, concise, and written in plain business language.

Final decision rule:
- Return TRUE if all applicable criteria are satisfied.
- Return FALSE if any applicable criterion fails.
- Return only TRUE or FALSE.
- Do not include explanations, markdown, or extra text.
""",
    model="databricks:/databricks-gpt-oss-20b",
    feedback_value_type=bool
)

print("Custom TRUE/FALSE judge created: restaurant_owner_judge")

Custom TRUE/FALSE judge created: restaurant_owner_judge


### 9.6. Run LLM Judge Evaluation

In [0]:
print("=" * 60)
print("Running LLM judge evaluation...")
print("=" * 60)

eval_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        restaurant_owner_judge
    ]
)

print("\nEvaluation complete.")
print("Review the MLflow evaluation results in the Experiments UI.")

Running LLM judge evaluation...


2026/06/22 17:22:52 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/06/22 17:22:52 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/5 [Elapsed: 00:00, Remaining: ?]

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


2026/06/22 17:23:13 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'restaurant_owner_judge': 4/5 failed. Check individual trace assessments for detailed error messages.
2026/06/22 17:23:14 WARNING mlflow.models.evaluation.utils.trace: Exception raised while calling autologging for spark: MLflow Spark dataset autologging is not supported on Databricks shared clusters or Databricks serverless clusters.



Evaluation complete.
Review the MLflow evaluation results in the Experiments UI.


---
## 10. Human Review Layer

This section implements the human evaluation component to validate LLM judge scores and compare model performance.

**Purpose**: Establish ground truth through human evaluation and measure agreement between human reviewers and the automated LLM judge.

**Process Overview**:
1. **Prepare Evaluation Materials**: Extract test cases, responses, and judge scores
2. **Distribute to Reviewers**: Each team member independently scores all 10 responses (5 questions × 2 models)
3. **Calculate Inter-Rater Agreement**: Measure consistency across human reviewers
4. **Consensus Discussion**: Resolve disagreements through team discussion
5. **Compare Human vs LLM Judge**: Analyze agreement between human consensus and automated scoring
6. **Document Findings**: Compile insights for presentation

**Evaluation Rubric**: Each response is scored PASS/FAIL on 7 dimensions:
- **Groundedness**: All facts verifiable from tool outputs
- **Answer Relevance**: Directly addresses the question
- **Restaurant Owner Usefulness**: Provides actionable business insight
- **Specificity**: Includes concrete numbers/themes/examples
- **Appropriate Tool Use**: Selected correct tool(s)
- **Out-of-Scope Handling**: Properly rejected irrelevant requests
- **Clarity**: Clear and understandable for a small business owner (organized, concise, plain language)

### 10.1. Prepare Evaluation Materials

Extract the 5 test cases with responses from both LLMs into a structured format for human review.

**Test Cases**:
1. **Business Lookup** (In-Scope): "How many reviews does Parma Cucina Italiana have, and what is its average rating?"
2. **Competitor Comparison** (In-Scope): "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?"
3. **Theme Retrieval** (In-Scope): "What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?"
4. **Out-of-Scope #1**: "What is the best recipe for homemade lasagna?"
5. **Out-of-Scope #2**: "Can you summarize reviews for a dentist in New York?"

Each test case includes:
- Original question
- Agent responses (Llama 3.3 70B and GPT OSS 20B)
- Tools called
- Expected tool usage
- LLM judge score (from Section 8 evaluation)

In [0]:
# Extract responses from the test cells (63, 65, 67, 69, 71)
# We'll structure this data for human review

import pandas as pd
from pyspark.sql import Row

# Define test cases with metadata
test_cases = [
    {
        "test_id": "TEST_01",
        "question": "How many reviews does Parma Cucina Italiana have, and what is its average rating?",
        "expected_tools": ["business_lookup"],
        "scope": "In-Scope",
        "query_type": "Business Lookup"
    },
    {
        "test_id": "TEST_02",
        "question": "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?",
        "expected_tools": ["competitor_comparison", "theme_retrieval"],
        "scope": "In-Scope",
        "query_type": "Competitor Comparison"
    },
    {
        "test_id": "TEST_03",
        "question": "What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?",
        "expected_tools": ["theme_retrieval"],
        "scope": "In-Scope",
        "query_type": "Theme Retrieval"
    },
    {
        "test_id": "TEST_04",
        "question": "What is the best recipe for homemade lasagna?",
        "expected_tools": [],
        "scope": "Out-of-Scope",
        "query_type": "Rejection Test"
    },
    {
        "test_id": "TEST_05",
        "question": "Can you summarize reviews for a dentist in New York?",
        "expected_tools": [],
        "scope": "Out-of-Scope",
        "query_type": "Rejection Test"
    }
]

print("=" * 60)
print("Test Cases Prepared for Human Review")
print("=" * 60)
print(f"Total test cases: {len(test_cases)}")
print(f"In-scope cases: {sum(1 for tc in test_cases if tc['scope'] == 'In-Scope')}")
print(f"Out-of-scope cases: {sum(1 for tc in test_cases if tc['scope'] == 'Out-of-Scope')}")

Test Cases Prepared for Human Review
Total test cases: 5
In-scope cases: 3
Out-of-scope cases: 2


In [0]:
# Re-run all test cases to capture responses and tool calls
# This ensures we have clean data for human review

from datetime import datetime
import json

evaluation_records = []

for test_case in test_cases:
    test_id = test_case["test_id"]
    question = test_case["question"]
    
    print(f"\n{'='*80}")
    print(f"{test_id}: {test_case['query_type']}")
    print(f"{'='*80}")
    print(f"Question: {question}\n")
    
    # Run with Llama 3.3 70B
    print("Running Llama 3.3 70B...")
    llama_response = AGENT_LLAMA_3_3_70B.predict(question, verbose=True)
    
    # Run with GPT OSS 20B
    print("\nRunning GPT OSS 20B...")
    gpt_response = AGENT_GPT_OSS_20B.predict(question, verbose=True)
    
    # Store both responses
    evaluation_records.append({
        "test_id": test_id,
        "question": question,
        "query_type": test_case["query_type"],
        "scope": test_case["scope"],
        "expected_tools": ", ".join(test_case["expected_tools"]) if test_case["expected_tools"] else "None (should reject)",
        "model": "Llama 3.3 70B",
        "response": llama_response
    })
    
    evaluation_records.append({
        "test_id": test_id,
        "question": question,
        "query_type": test_case["query_type"],
        "scope": test_case["scope"],
        "expected_tools": ", ".join(test_case["expected_tools"]) if test_case["expected_tools"] else "None (should reject)",
        "model": "GPT OSS 20B",
        "response": gpt_response
    })
    
    print("\n" + "-"*80)

print(f"\n{'='*80}")
print(f"Captured {len(evaluation_records)} evaluation records")
print(f"({len(test_cases)} questions × 2 models)")
print(f"{'='*80}")


TEST_01: Business Lookup
Question: How many reviews does Parma Cucina Italiana have, and what is its average rating?

Running Llama 3.3 70B...
Calling tool: business_lookup
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'limit': 1}

Running GPT OSS 20B...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ma Cucina Italiana.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Calling tool: business_lookup
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'limit': 1}


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ded to two decimals).'}], input_type=list])
  return self.__pydantic_serializer__.to_python(



--------------------------------------------------------------------------------

TEST_02: Competitor Comparison
Question: Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?

Running Llama 3.3 70B...
Calling tool: competitor_comparison
Arguments: {'restaurant_1': 'Parma Cucina Italiana', 'restaurant_2': 'Cesarina', 'limit_per_name': 1}
Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Cesarina', 'theme_query': 'food quality and authentic Italian food', 'num_results': 5}
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

Running GPT OSS 20B...


...

/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...so. We\'ll produce.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'theme_query': 'pasta quality and Italian authenticity', 'num_results': 3}
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

--------------------------------------------------------------------------------

TEST_04: Rejection Test
Question: What is the best recipe for homemade lasagna?

Running Llama 3.3 70B...

Running GPT OSS 20B...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's... Italian restaurants.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(



--------------------------------------------------------------------------------

TEST_05: Rejection Test
Question: Can you summarize reviews for a dentist in New York?

Running Llama 3.3 70B...

Running GPT OSS 20B...

--------------------------------------------------------------------------------

Captured 10 evaluation records
(5 questions × 2 models)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-51018d6f-cce6-49d6-a98c-28d058e703fb/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...d be happy to assist!'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


In [0]:
# Convert to pandas DataFrame first (handles mixed types better)
# Then convert to Spark DataFrame for display and storage
import pandas as pd

# Ensure all response values are strings
for record in evaluation_records:
    record["response"] = str(record["response"])

# Create pandas DataFrame
evaluation_pandas_df = pd.DataFrame(evaluation_records)

# Convert to Spark DataFrame
evaluation_df = spark.createDataFrame(evaluation_pandas_df)

print("=" * 60)
print("Human Review Dataset Created")
print("=" * 60)

# Display the structured dataset
display(evaluation_df.select(
    "test_id",
    "question",
    "query_type",
    "scope",
    "expected_tools",
    "model",
    "response"
).orderBy("test_id", "model"))

print(f"\nTotal records: {evaluation_df.count()}")
print("\nReady for export to human review spreadsheet.")

Human Review Dataset Created


test_id,question,query_type,scope,expected_tools,model,response
TEST_01,"How many reviews does Parma Cucina Italiana have, and what is its average rating?",Business Lookup,In-Scope,business_lookup,GPT OSS 20B,"[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'We have results: review_count=552, avg_rating=4.670289855072464. The user: ""How many reviews does Parma Cucina Italiana have, and what is its average rating?"" So answer: 552 reviews, average rating about 4.67. Provide source.'}]}, {'type': 'text', 'text': 'Parma\u202fCucina\u202fItaliana has **552 reviews** and an average rating of **about\u202f4.67\u202fstars** (rounded to two decimals).'}]"
TEST_01,"How many reviews does Parma Cucina Italiana have, and what is its average rating?",Business Lookup,In-Scope,business_lookup,Llama 3.3 70B,Parma Cucina Italiana has an average rating of 4.67 and 552 reviews.
TEST_02,Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?,Competitor Comparison,In-Scope,"competitor_comparison, theme_retrieval",GPT OSS 20B,"[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'Both have positive. So not helpful. We might need theme on ""price/value"" or ""ambiance"". Let\'s try ""value for money"".\n\n{""restaurant_name"":""Parma Cucina Italiana"",""theme_query"":""value for money"",""num_results"":3}'}]}]"
TEST_02,Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?,Competitor Comparison,In-Scope,"competitor_comparison, theme_retrieval",Llama 3.3 70B,"Based on the comparison of Parma Cucina Italiana and Cesarina, Cesarina appears to have a slightly higher average rating (4.84 vs 4.67) and more reviews with a 5-star rating (463 vs 429). The theme retrieval results for Cesarina also suggest that customers praise the restaurant for its authentic Italian food, with many reviewers mentioning the freshness and quality of the ingredients. Some reviewers also complimented the service, saying it was great or terrific. Overall, it seems that Cesarina does better in terms of food quality and authenticity, as well as customer satisfaction."
TEST_03,What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?,Theme Retrieval,In-Scope,theme_retrieval,GPT OSS 20B,"**What customers say about pasta quality & Italian authenticity at Parma Cucina Italiana** | Rating | Review excerpt | Theme signal | |--------|----------------|--------------| | 5‑star | “Authentic Italian” | High‑quality, authentic pasta | | 4‑star | “Very good and authentic Italian” | Positive experience, still authentic | | 5‑star | “Great Italian food” | Praise for authenticity and overall quality | These snippets come from recent reviews that specifically mention pasta quality and authenticity. Overall, customers consistently note that Parma Cucina Italiana serves genuine, high‑quality Italian pasta."
TEST_03,What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?,Theme Retrieval,In-Scope,theme_retrieval,Llama 3.3 70B,"Based on the review excerpts retrieved, customers are praising Parma Cucina Italiana for its authentic Italian cuisine, with one reviewer explicitly stating that the food is ""Authentic Italian"" and another describing it as ""Very good and authentic Italian"". A third reviewer simply states that they had ""Great Italian food"". These reviews suggest that Parma Cucina Italiana is successful in providing an authentic Italian dining experience, which is a key aspect of its appeal to customers. Overall, the feedback on pasta quality and Italian authenticity at Parma Cucina Italiana is positive, with reviewers appreciating the restaurant's commitment to traditional Italian flavors and cuisine."
TEST_04,What is the best recipe for homemade lasagna?,Rejection Test,Out-of-Scope,None (should reject),GPT OSS 20B,"[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'User asks: ""What is the best 


Total records: 10

Ready for export to human review spreadsheet.


In [0]:
# Save the evaluation dataset to Unity Catalog
# This allows team members to access it for review

REVIEW_TABLE = f"{CATALOG}.{SCHEMA}.human_review_dataset"

evaluation_df.write.mode("overwrite").saveAsTable(REVIEW_TABLE)

print("=" * 60)
print("Human Review Dataset Saved")
print("=" * 60)
print(f"Table: {REVIEW_TABLE}")
print(f"\nTeam members can access this table for human evaluation.")
print(f"\nNext Steps:")
print("1. Export to Google Sheets or Excel for review")
print("2. Add review columns: human_score, groundedness, relevance, usefulness, specificity, tool_use, scope_handling, comments")
print("3. Distribute to all team members for independent review")
print("4. Collect reviews and calculate inter-rater agreement")

Human Review Dataset Saved
Table: main.aai510_final_agent.human_review_dataset

Team members can access this table for human evaluation.

Next Steps:
1. Export to Google Sheets or Excel for review
2. Add review columns: human_score, groundedness, relevance, usefulness, specificity, tool_use, scope_handling, comments
3. Distribute to all team members for independent review
4. Collect reviews and calculate inter-rater agreement


In [0]:
# Create a template with review columns for human evaluators
# This can be exported to a spreadsheet

from pyspark.sql.functions import lit

review_template_df = evaluation_df.select(
    "test_id",
    "question",
    "query_type",
    "scope",
    "expected_tools",
    "model",
    "response"
)

print("=" * 60)
print("Human Review Template Created")
print("=" * 60)
print("\nTemplate includes the following review columns:")
print("- llm_judge_score: PASS/FAIL from LLM judge")
print("- human_overall_score: PASS/FAIL from human reviewer")
print("- groundedness: PASS/FAIL (facts verifiable from tool outputs)")
print("- answer_relevance: PASS/FAIL (directly answers question)")
print("- usefulness: PASS/FAIL (provides actionable insight)")
print("- specificity: PASS/FAIL (includes concrete details)")
print("- tool_use: PASS/FAIL (selected appropriate tool)")
print("- scope_handling: PASS/FAIL (proper rejection of out-of-scope)")
print("- clarity: PASS/FAIL (clear and understandable for business owner)")
print("- comments: Free text for reviewer notes")
print("- reviewer_name: Name of the team member conducting review")

# Display sample
print("\nSample Template (first 2 records):")
display(review_template_df)

Human Review Template Created

Template includes the following review columns:
- llm_judge_score: PASS/FAIL from LLM judge
- human_overall_score: PASS/FAIL from human reviewer
- groundedness: PASS/FAIL (facts verifiable from tool outputs)
- answer_relevance: PASS/FAIL (directly answers question)
- usefulness: PASS/FAIL (provides actionable insight)
- specificity: PASS/FAIL (includes concrete details)
- tool_use: PASS/FAIL (selected appropriate tool)
- scope_handling: PASS/FAIL (proper rejection of out-of-scope)
- clarity: PASS/FAIL (clear and understandable for business owner)
- comments: Free text for reviewer notes
- reviewer_name: Name of the team member conducting review

Sample Template (first 2 records):


test_id,question,query_type,scope,expected_tools,model,response
TEST_01,"How many reviews does Parma Cucina Italiana have, and what is its average rating?",Business Lookup,In-Scope,business_lookup,Llama 3.3 70B,Parma Cucina Italiana has an average rating of 4.67 and 552 reviews.
TEST_01,"How many reviews does Parma Cucina Italiana have, and what is its average rating?",Business Lookup,In-Scope,business_lookup,GPT OSS 20B,"[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'We have results: review_count=552, avg_rating=4.670289855072464. The user: ""How many reviews does Parma Cucina Italiana have, and what is its average rating?"" So answer: 552 reviews, average rating about 4.67. Provide source.'}]}, {'type': 'text', 'text': 'Parma\u202fCucina\u202fItaliana has **552 reviews** and an average rating of **about\u202f4.67\u202fstars** (rounded to two decimals).'}]"
TEST_02,Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?,Competitor Comparison,In-Scope,"competitor_comparison, theme_retrieval",Llama 3.3 70B,"Based on the comparison of Parma Cucina Italiana and Cesarina, Cesarina appears to have a slightly higher average rating (4.84 vs 4.67) and more reviews with a 5-star rating (463 vs 429). The theme retrieval results for Cesarina also suggest that customers praise the restaurant for its authentic Italian food, with many reviewers mentioning the freshness and quality of the ingredients. Some reviewers also complimented the service, saying it was great or terrific. Overall, it seems that Cesarina does better in terms of food quality and authenticity, as well as customer satisfaction."
TEST_02,Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?,Competitor Comparison,In-Scope,"competitor_comparison, theme_retrieval",GPT OSS 20B,"[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'Both have positive. So not helpful. We might need theme on ""price/value"" or ""ambiance"". Let\'s try ""value for money"".\n\n{""restaurant_name"":""Parma Cucina Italiana"",""theme_query"":""value for money"",""num_results"":3}'}]}]"
TEST_03,What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?,Theme Retrieval,In-Scope,theme_retrieval,Llama 3.3 70B,"Based on the review excerpts retrieved, customers are praising Parma Cucina Italiana for its authentic Italian cuisine, with one reviewer explicitly stating that the food is ""Authentic Italian"" and another describing it as ""Very good and authentic Italian"". A third reviewer simply states that they had ""Great Italian food"". These reviews suggest that Parma Cucina Italiana is successful in providing an authentic Italian dining experience, which is a key aspect of its appeal to customers. Overall, the feedback on pasta quality and Italian authenticity at Parma Cucina Italiana is positive, with reviewers appreciating the restaurant's commitment to traditional Italian flavors and cuisine."
TEST_03,What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?,Theme Retrieval,In-Scope,theme_retrieval,GPT OSS 20B,"**What customers say about pasta quality & Italian authenticity at Parma Cucina Italiana** | Rating | Review excerpt | Theme signal | |--------|----------------|--------------| | 5‑star | “Authentic Italian” | High‑quality, authentic pasta | | 4‑star | “Very good and authentic Italian” | Positive experience, still authentic | | 5‑star | “Great Italian food” | Praise for authenticity and overall quality | These snippets come from recent reviews that specifically mention pasta quality and authenticity. Overall, customers consistently note that Parma Cucina Italiana serves genuine, high‑quality Italian pasta."
TEST_04,What is the best recipe for homemade lasagna?,Rejection Test,Out-of-Scope,None (should reject),Llama 3.3 70B,"I can only help with San Diego Italian restaurant comparison questions using the available data. 

### 10.2. Team Review for Responses

**Quick Rubric Reference for Reviewers**:

| Dimension | PASS Criteria | FAIL Example |
|-----------|---------------|---------------|
| **Groundedness** | All facts verifiable from tool outputs | "Parma has 600 reviews" (actual: 552) |
| **Answer Relevance** | Directly answers the question | Generic response, misses the question |
| **Usefulness** | Provides actionable business insight | Just dumps data without interpretation |
| **Specificity** | Includes concrete numbers/themes | Vague: "customers seem happy" |
| **Tool Use** | Selected appropriate tool(s) | Used wrong tool or skipped necessary tools |
| **Scope Handling** | In-scope: answered; Out-of-scope: politely rejected | Attempted to answer recipe question |
| **Clarity** | Clear, organized, concise, plain business language | Confusing, disorganized, overly technical jargon |

**Scoring Rule**: PASS = all 7 criteria met, FAIL = any criterion failed

### Human Review Table - Team Evaluation

**Instructions**: Each of the three team members should independently review all 5 test questions and both model responses (Llama 3.3 70B and GPT OSS 20B). Fill in your scores below.

**Scoring**: Use PASS or FAIL for each dimension. Overall = PASS only if ALL dimensions pass.

---

#### Reviewer 1: Brandon Wirgau

| Test ID | Model | Question Summary | Groundedness | Answer Relevance | Usefulness | Specificity | Tool Use | Scope Handling | Clarity | Overall Score | Comments |
|---------|-------|------------------|--------------|------------------|------------|-------------|----------|----------------|---------|---------------|----------|
| TEST_01 | Llama 3.3 70B | Parma review count & rating | P | P | P | P | P | P | P | P |
| TEST_01 | GPT OSS 20B | Parma review count & rating | P | P | P | P | P | P | P | P |
| TEST_02 | Llama 3.3 70B | Compare Parma vs Cesarina | P | P | P | P | P | P | P | P |
| TEST_02 | GPT OSS 20B | Compare Parma vs Cesarina | F | F | F | F | F | F | F | F | Gave recommendation about retrieving additional data instead of providing a direct answer.
| TEST_03 | Llama 3.3 70B | Pasta quality themes | P | P | P | P | P | P | P | P |
| TEST_03 | GPT OSS 20B | Pasta quality themes | P | P | P | P | P | P | P | P |
| TEST_04 | Llama 3.3 70B | Lasagna recipe (out-of-scope) | P | P | P | P | P | P | P | P | Stayed in the scope of the question and did not try to answer anything out of scope.
| TEST_04 | GPT OSS 20B | Lasagna recipe (out-of-scope) | P | P | P | P | P | P | P | P |
| TEST_05 | Llama 3.3 70B | Dentist in NY (out-of-scope) | P | P | P | P | P | P | P | P |
| TEST_05 | GPT OSS 20B | Dentist in NY (out-of-scope) | P | P | P | P | P | P | P | P |

---

#### Reviewer 2: Erika Gallegos

| Test ID | Model | Question Summary | Groundedness | Answer Relevance | Usefulness | Specificity | Tool Use | Scope Handling | Clarity | Overall Score | Comments |
|---------|-------|------------------|--------------|------------------|------------|-------------|----------|----------------|---------|---------------|----------|
| TEST_01 | Llama 3.3 70B | Parma review count & rating | P | P | P | P | P | P | P | P |
| TEST_01 | GPT OSS 20B | Parma review count & rating | P | P | P | P | P | P | P | P |
| TEST_02 | Llama 3.3 70B | Compare Parma vs Cesarina | P | P | P | P | P | P | P | P |
| TEST_02 | GPT OSS 20B | Compare Parma vs Cesarina | F | F | F | F | F | F | F | F | Never reached an answer, just kept calling tools then timed out.
| TEST_03 | Llama 3.3 70B | Pasta quality themes | P | P | P | P | P | P | P | P |
| TEST_03 | GPT OSS 20B | Pasta quality themes | P | P | P | P | P | P | P | P |
| TEST_04 | Llama 3.3 70B | Lasagna recipe (out-of-scope) | P | P | P | P | P | P | P | P |
| TEST_04 | GPT OSS 20B | Lasagna recipe (out-of-scope) | P | P | P | P | P | P | P | P |
| TEST_05 | Llama 3.3 70B | Dentist in NY (out-of-scope) | P | P | P | P | P | P | P | P |
| TEST_05 | GPT OSS 20B | Dentist in NY (out-of-scope) | P | P | P | P | P | P | P | P |

---

#### Reviewer 3: Eric Hernandez

| Test ID | Model | Question Summary | Groundedness | Answer Relevance | Usefulness | Specificity | Tool Use | Scope Handling | Clarity | Overall Score | Comments |
|---------|-------|------------------|--------------|------------------|------------|-------------|----------|----------------|---------|---------------|----------|
| TEST_01 | Llama 3.3 70B | Parma review count & rating | P | P | P | P | P | P | P | P | Correct tool, good response
| TEST_01 | GPT OSS 20B | Parma review count & rating | P | P | P | P | P | P | P | P | Correct tool, acceptable response.
| TEST_02 | Llama 3.3 70B | Compare Parma vs Cesarina | P | P | P | P | P | P | P | P | Good tool usage, good response
| TEST_02 | GPT OSS 20B | Compare Parma vs Cesarina | F | F | F | F | F | F | F | F | Reached maximum number of tool calls. No response
| TEST_03 | Llama 3.3 70B | Pasta quality themes | P | P | P | P | P | P | P | P | Good tool usage and response
| TEST_03 | GPT OSS 20B | Pasta quality themes | P | P | P | P | P | P | P | P | Good response
| TEST_04 | Llama 3.3 70B | Lasagna recipe (out-of-scope) | P | P | P | P | P | P | P | P | Correctly explained question was out of scope
| TEST_04 | GPT OSS 20B | Lasagna recipe (out-of-scope) | P | P | P | P | P | P | P | P | Good out of scope response
| TEST_05 | Llama 3.3 70B | Dentist in NY (out-of-scope) | P | P | P | P | P | P | P | P | Correctly explained question was out of scope.
| TEST_05 | GPT OSS 20B | Dentist in NY (out-of-scope) | P | P | P | P | P | P | P | P | Correct respose

---

**After all reviews are complete**:
1. Calculate inter-rater agreement (Cohen's Kappa or Fleiss' Kappa)
2. Identify disagreements for consensus discussion
3. Compare human consensus vs LLM judge scores
4. Document findings for presentation

---
## 11. ROI Analysis - Model Comparison

This section calculates the Return on Investment (ROI) for both LLMs by comparing:
1. **Cost**: Token usage and pricing
2. **Quality**: Evaluation scores from LLM judge and human review
3. **Efficiency**: Response time and latency
4. **Business Value**: Monetary value of correct answers vs. cost of errors

**Goal**: Determine which model provides the best value for a restaurant owner use case.

**Analysis Framework**:
- **ROI Formula**: ((Total Value - Total Cost) / Total Cost) × 100%
- **Value Sources**: Correct answers, proper rejections, fast responses
- **Cost Sources**: Token consumption, compute time, error penalties
- **Decision Criteria**: Highest ROI with acceptable quality threshold (>80% pass rate)

### 11.1. Extract Token Usage from MLflow Traces

Analyze token consumption for each model across the 5 test queries.

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

# Initialize MLflow client
client = MlflowClient()

# Get the experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# Get all runs from the evaluation
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=100
)

print("="*60)
print("MLflow Runs Found")
print("="*60)
print(f"Total runs: {len(runs)}")

# Display run metadata
if len(runs) > 0:
    display(runs[["run_id", "start_time", "tags.mlflow.runName"]].head(10))
else:
    print("No runs found. Make sure Section 8 evaluation completed successfully.")

MLflow Runs Found
Total runs: 4


run_id,start_time,tags.mlflow.runName
e271631bee39428fa986634206e0d966,2026-06-22T17:22:52.715Z,adaptable-jay-26
2476ca4402624a048e844fc5f5060603,2026-06-22T02:13:14.721Z,gaudy-perch-50
f0da6cbc844e40ebb636b49d69942fd1,2026-06-17T04:02:35.713Z,tasteful-roo-974
59324f288da0499cb8cdfff011e28537,2026-06-17T04:01:52.790Z,funny-fly-628


In [0]:
# Extract actual token usage from MLflow traces
import pandas as pd

print("Extracting token usage from MLflow runs...")
print(f"Found {len(runs)} runs in experiment\n")

# Initialize token usage dictionary
token_usage = {
    "Llama 3.3 70B": {},
    "GPT OSS 20B": {}
}

# Map of questions to test IDs
question_to_test = {
    "How many reviews does Parma Cucina Italiana have": "TEST_01",
    "Compare Parma Cucina Italiana and Cesarina": "TEST_02",
    "pasta quality and Italian authenticity": "TEST_03",
    "best recipe for homemade lasagna": "TEST_04",
    "dentist in New York": "TEST_05"
}

# Try to extract token usage from MLflow runs
tokens_extracted = False

for idx, row in runs.iterrows():
    # Try to get token counts from metrics
    input_tokens = row.get('metrics.token_usage.prompt_tokens', None)
    output_tokens = row.get('metrics.token_usage.completion_tokens', None)
    total_tokens = row.get('metrics.token_usage.total_tokens', None)
    
    # Alternative metric names
    if pd.isna(input_tokens):
        input_tokens = row.get('metrics.prompt_tokens', None)
    if pd.isna(output_tokens):
        output_tokens = row.get('metrics.completion_tokens', None)
    if pd.isna(total_tokens):
        total_tokens = row.get('metrics.total_tokens', None)
    
    # If we have token data
    if pd.notna(total_tokens) or (pd.notna(input_tokens) and pd.notna(output_tokens)):
        # Calculate missing values if needed
        if pd.notna(input_tokens) and pd.notna(output_tokens) and pd.isna(total_tokens):
            total_tokens = input_tokens + output_tokens
        elif pd.notna(total_tokens):
            if pd.isna(input_tokens):
                input_tokens = int(total_tokens * 0.8)  # Estimate
            if pd.isna(output_tokens):
                output_tokens = int(total_tokens * 0.2)  # Estimate
        
        # Extract model name from params
        model_name = None
        if 'params.model' in row and pd.notna(row['params.model']):
            model_param = str(row['params.model'])
            if 'llama-3-3-70b' in model_param.lower():
                model_name = "Llama 3.3 70B"
            elif 'gpt-oss-20b' in model_param.lower():
                model_name = "GPT OSS 20B"
        
        # Try to match question to test ID from tags
        test_id = None
        if 'tags.mlflow.note.content' in row and pd.notna(row['tags.mlflow.note.content']):
            note = str(row['tags.mlflow.note.content'])
            for q_part, tid in question_to_test.items():
                if q_part.lower() in note.lower():
                    test_id = tid
                    break
        
        if model_name and test_id:
            token_usage[model_name][test_id] = {
                "input": int(input_tokens),
                "output": int(output_tokens),
                "total": int(total_tokens)
            }
            tokens_extracted = True

# If we couldn't extract from MLflow, use estimates
if not tokens_extracted or (not token_usage["Llama 3.3 70B"] and not token_usage["GPT OSS 20B"]):
    print("⚠️  Could not extract token usage from MLflow traces.")
    print("Using estimated values based on typical usage patterns.\n")
    
    token_usage = {
        "Llama 3.3 70B": {
            "TEST_01": {"input": 850, "output": 120, "total": 970},
            "TEST_02": {"input": 950, "output": 280, "total": 1230},
            "TEST_03": {"input": 920, "output": 350, "total": 1270},
            "TEST_04": {"input": 830, "output": 80, "total": 910},
            "TEST_05": {"input": 840, "output": 85, "total": 925}
        },
        "GPT OSS 20B": {
            "TEST_01": {"input": 820, "output": 100, "total": 920},
            "TEST_02": {"input": 930, "output": 250, "total": 1180},
            "TEST_03": {"input": 910, "output": 320, "total": 1230},
            "TEST_04": {"input": 810, "output": 70, "total": 880},
            "TEST_05": {"input": 820, "output": 75, "total": 895}
        }
    }
else:
    print("✓ Successfully extracted token usage from MLflow traces\n")

# Calculate totals
token_summary = {}

for model, tests in token_usage.items():
    total_input = sum(test["input"] for test in tests.values())
    total_output = sum(test["output"] for test in tests.values())
    total_tokens = sum(test["total"] for test in tests.values())
    avg_per_query = total_tokens / len(tests)
    
    token_summary[model] = {
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
        "total_tokens": total_tokens,
        "avg_tokens_per_query": avg_per_query,
        "num_queries": len(tests)
    }

print("="*60)
print("Token Usage Summary")
print("="*60)

for model, stats in token_summary.items():
    print(f"\n{model}:")
    print(f"  Total input tokens:  {stats['total_input_tokens']:,}")
    print(f"  Total output tokens: {stats['total_output_tokens']:,}")
    print(f"  Total tokens:        {stats['total_tokens']:,}")
    print(f"  Avg per query:       {stats['avg_tokens_per_query']:.1f}")
    print(f"  Number of queries:   {stats['num_queries']}")

Extracting token usage from MLflow runs...
Found 4 runs in experiment

⚠️  Could not extract token usage from MLflow traces.
Using estimated values based on typical usage patterns.

Token Usage Summary

Llama 3.3 70B:
  Total input tokens:  4,390
  Total output tokens: 915
  Total tokens:        5,305
  Avg per query:       1061.0
  Number of queries:   5

GPT OSS 20B:
  Total input tokens:  4,290
  Total output tokens: 815
  Total tokens:        5,105
  Avg per query:       1021.0
  Number of queries:   5


### 11.2. Define Pricing and Calculate Costs

Apply Databricks Model Serving pricing to calculate cost per query and total evaluation cost.

**Databricks Foundation Model API Pricing** (as of 2024):
- Prices vary by model size and type
- Typically charged per 1M tokens (input and output separate)
- Note: Use actual pricing from your Databricks workspace for accurate ROI

In [0]:
# Databricks Foundation Model API pricing
# Rates as of June 18, 2026: https://www.databricks.com/product/pricing/foundation-model-serving

pricing = {
    "Llama 3.3 70B": {
        "input_per_1m": 0.50,   # $0.50 per 1M input tokens
        "output_per_1m": 1.50,  # $1.50 per 1M output tokens
        "model_size": "70B"
    },
    "GPT OSS 20B": {
        "input_per_1m": 0.07,   # $0.40 per 1M input tokens
        "output_per_1m": 0.30,  # $1.20 per 1M output tokens
        "model_size": "20B"
    }
}

print("="*60)
print("Model Pricing Configuration")
print("="*60)

for model, rates in pricing.items():
    print(f"\n{model} ({rates['model_size']}):")
    print(f"  Input:  ${rates['input_per_1m']}/1M tokens")
    print(f"  Output: ${rates['output_per_1m']}/1M tokens")

Model Pricing Configuration

Llama 3.3 70B (70B):
  Input:  $0.5/1M tokens
  Output: $1.5/1M tokens

GPT OSS 20B (20B):
  Input:  $0.07/1M tokens
  Output: $0.3/1M tokens


In [0]:
# Calculate cost for each model
cost_analysis = {}

for model in token_summary.keys():
    input_tokens = token_summary[model]["total_input_tokens"]
    output_tokens = token_summary[model]["total_output_tokens"]
    
    input_cost = (input_tokens / 1_000_000) * pricing[model]["input_per_1m"]
    output_cost = (output_tokens / 1_000_000) * pricing[model]["output_per_1m"]
    total_cost = input_cost + output_cost
    
    cost_per_query = total_cost / token_summary[model]["num_queries"]
    
    # Project monthly costs (assuming 1000 queries/month)
    monthly_cost_1k = cost_per_query * 1000
    monthly_cost_10k = cost_per_query * 10000
    
    cost_analysis[model] = {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
        "cost_per_query": cost_per_query,
        "monthly_cost_1k_queries": monthly_cost_1k,
        "monthly_cost_10k_queries": monthly_cost_10k
    }

print("="*60)
print("Cost Analysis")
print("="*60)

for model, costs in cost_analysis.items():
    print(f"\n{model}:")
    print(f"  Input cost:              ${costs['input_cost']:.6f}")
    print(f"  Output cost:             ${costs['output_cost']:.6f}")
    print(f"  Total evaluation cost:   ${costs['total_cost']:.6f}")
    print(f"  Cost per query:          ${costs['cost_per_query']:.6f}")
    print(f"  Monthly (1K queries):    ${costs['monthly_cost_1k_queries']:.2f}")
    print(f"  Monthly (10K queries):   ${costs['monthly_cost_10k_queries']:.2f}")

# Calculate cost difference
cost_diff = cost_analysis["Llama 3.3 70B"]["cost_per_query"] - cost_analysis["GPT OSS 20B"]["cost_per_query"]
cost_diff_pct = (cost_diff / cost_analysis["GPT OSS 20B"]["cost_per_query"]) * 100

print(f"\n{'='*60}")
print(f"Cost Comparison:")
if cost_diff > 0:
    print(f"  Llama 3.3 70B is ${cost_diff:.6f} ({cost_diff_pct:.1f}%) MORE expensive per query")
else:
    print(f"  Llama 3.3 70B is ${abs(cost_diff):.6f} ({abs(cost_diff_pct):.1f}%) LESS expensive per query")

Cost Analysis

Llama 3.3 70B:
  Input cost:              $0.002195
  Output cost:             $0.001373
  Total evaluation cost:   $0.003567
  Cost per query:          $0.000713
  Monthly (1K queries):    $0.71
  Monthly (10K queries):   $7.13

GPT OSS 20B:
  Input cost:              $0.000300
  Output cost:             $0.000244
  Total evaluation cost:   $0.000545
  Cost per query:          $0.000109
  Monthly (1K queries):    $0.11
  Monthly (10K queries):   $1.09

Cost Comparison:
  Llama 3.3 70B is $0.000605 (554.8%) MORE expensive per query


### 11.3. Aggregate Quality Scores

Collect quality metrics from LLM judge evaluation and human review (when available).

In [0]:
# Quality scores based on test results (Section 7 & 8)
# These should be extracted from actual evaluation results

quality_scores = {
    "Llama 3.3 70B": {
        "TEST_01": {"pass": True, "groundedness": True, "relevance": True, "usefulness": True, "specificity": True, "tool_use": True, "scope": True},
        "TEST_02": {"pass": True, "groundedness": True, "relevance": True, "usefulness": True, "specificity": True, "tool_use": True, "scope": True},
        "TEST_03": {"pass": True, "groundedness": True, "relevance": True, "usefulness": True, "specificity": True, "tool_use": True, "scope": True},
        "TEST_04": {"pass": True, "groundedness": True, "relevance": True, "usefulness": True, "specificity": True, "tool_use": True, "scope": True},  # Proper rejection
        "TEST_05": {"pass": True, "groundedness": True, "relevance": True, "usefulness": True, "specificity": True, "tool_use": True, "scope": True}   # Proper rejection
    },
    "GPT OSS 20B": {
        "TEST_01": {"pass": True, "groundedness": True, "relevance": True, "usefulness": True, "specificity": True, "tool_use": True, "scope": True},
        "TEST_02": {"pass": True, "groundedness": True, "relevance": True, "usefulness": True, "specificity": True, "tool_use": True, "scope": True},
        "TEST_03": {"pass": False, "groundedness": True, "relevance": True, "usefulness": False, "specificity": False, "tool_use": True, "scope": True},  # Lower quality theme retrieval
        "TEST_04": {"pass": True, "groundedness": True, "relevance": True, "usefulness": True, "specificity": True, "tool_use": True, "scope": True},   # Proper rejection
        "TEST_05": {"pass": False, "groundedness": True, "relevance": False, "usefulness": False, "specificity": True, "tool_use": True, "scope": False}  # Weaker rejection
    }
}

# Calculate pass rates
quality_summary = {}

for model, tests in quality_scores.items():
    total_tests = len(tests)
    passed_tests = sum(1 for test in tests.values() if test["pass"])
    pass_rate = (passed_tests / total_tests) * 100
    
    # Calculate dimension-specific pass rates
    dimensions = ["groundedness", "relevance", "usefulness", "specificity", "tool_use", "scope"]
    dimension_rates = {}
    for dim in dimensions:
        dim_passed = sum(1 for test in tests.values() if test[dim])
        dimension_rates[dim] = (dim_passed / total_tests) * 100
    
    quality_summary[model] = {
        "total_tests": total_tests,
        "passed_tests": passed_tests,
        "failed_tests": total_tests - passed_tests,
        "pass_rate": pass_rate,
        "dimension_rates": dimension_rates
    }

print("="*60)
print("Quality Summary - LLM Judge Scores")
print("="*60)

for model, stats in quality_summary.items():
    print(f"\n{model}:")
    print(f"  Total tests:      {stats['total_tests']}")
    print(f"  Passed:           {stats['passed_tests']}")
    print(f"  Failed:           {stats['failed_tests']}")
    print(f"  Overall pass rate: {stats['pass_rate']:.1f}%")
    print(f"\n  Dimension Pass Rates:")
    for dim, rate in stats['dimension_rates'].items():
        print(f"    {dim.capitalize():15s}: {rate:.1f}%")

Quality Summary - LLM Judge Scores

Llama 3.3 70B:
  Total tests:      5
  Passed:           5
  Failed:           0
  Overall pass rate: 100.0%

  Dimension Pass Rates:
    Groundedness   : 100.0%
    Relevance      : 100.0%
    Usefulness     : 100.0%
    Specificity    : 100.0%
    Tool_use       : 100.0%
    Scope          : 100.0%

GPT OSS 20B:
  Total tests:      5
  Passed:           3
  Failed:           2
  Overall pass rate: 60.0%

  Dimension Pass Rates:
    Groundedness   : 100.0%
    Relevance      : 80.0%
    Usefulness     : 60.0%
    Specificity    : 80.0%
    Tool_use       : 100.0%
    Scope          : 80.0%


### 11.4. Calculate Business Value

Assign monetary value to correct answers and penalties for errors based on restaurant owner use case.

In [0]:
# Business value framework for restaurant owner
value_framework = {
    "correct_answer_value": {
        "business_lookup": 10,        # Basic info retrieval
        "competitor_comparison": 50,  # Strategic insight
        "theme_retrieval": 100,       # Actionable feedback
        "proper_rejection": 5          # Prevents misinformation
    },
    "error_cost": {
        "wrong_metrics": -50,          # Bad business decisions
        "failed_rejection": -30,       # User confusion
        "incomplete_answer": -20       # Missing information
    },
    "efficiency_bonus": {
        "fast_response": 10,           # <3 seconds
        "slow_response_penalty": -5    # >10 seconds
    }
}

# Map test cases to value categories
test_value_mapping = {
    "TEST_01": "business_lookup",
    "TEST_02": "competitor_comparison",
    "TEST_03": "theme_retrieval",
    "TEST_04": "proper_rejection",
    "TEST_05": "proper_rejection"
}

print("="*60)
print("Business Value Framework")
print("="*60)
print("\nCorrect Answer Values:")
for category, value in value_framework["correct_answer_value"].items():
    print(f"  {category:25s}: ${value}")
print("\nError Costs:")
for category, cost in value_framework["error_cost"].items():
    print(f"  {category:25s}: ${cost}")
print("\nEfficiency Bonuses/Penalties:")
for category, value in value_framework["efficiency_bonus"].items():
    print(f"  {category:25s}: ${value}")

Business Value Framework

Correct Answer Values:
  business_lookup          : $10
  competitor_comparison    : $50
  theme_retrieval          : $100
  proper_rejection         : $5

Error Costs:
  wrong_metrics            : $-50
  failed_rejection         : $-30
  incomplete_answer        : $-20

Efficiency Bonuses/Penalties:
  fast_response            : $10
  slow_response_penalty    : $-5


In [0]:
# Calculate total value generated by each model
value_analysis = {}

for model, tests in quality_scores.items():
    total_value = 0
    value_breakdown = []
    
    for test_id, scores in tests.items():
        test_type = test_value_mapping[test_id]
        base_value = value_framework["correct_answer_value"][test_type]
        
        if scores["pass"]:
            # Full value for passing tests
            test_value = base_value
            value_breakdown.append({
                "test_id": test_id,
                "type": test_type,
                "value": test_value,
                "reason": "Passed all criteria"
            })
        else:
            # Partial value or penalties for failing tests
            if test_type in ["business_lookup", "competitor_comparison", "theme_retrieval"]:
                # Failed in-scope query - apply incomplete answer penalty
                test_value = value_framework["error_cost"]["incomplete_answer"]
                value_breakdown.append({
                    "test_id": test_id,
                    "type": test_type,
                    "value": test_value,
                    "reason": "Failed quality criteria"
                })
            else:
                # Failed rejection - apply rejection penalty
                test_value = value_framework["error_cost"]["failed_rejection"]
                value_breakdown.append({
                    "test_id": test_id,
                    "type": test_type,
                    "value": test_value,
                    "reason": "Failed to reject out-of-scope"
                })
        
        total_value += test_value
    
    # Add efficiency bonuses (assume fast responses for this analysis)
    efficiency_bonus = 5 * value_framework["efficiency_bonus"]["fast_response"]
    total_value += efficiency_bonus
    
    value_analysis[model] = {
        "total_value": total_value,
        "base_value": total_value - efficiency_bonus,
        "efficiency_bonus": efficiency_bonus,
        "breakdown": value_breakdown
    }

print("="*60)
print("Business Value Analysis")
print("="*60)

for model, analysis in value_analysis.items():
    print(f"\n{model}:")
    print(f"  Base value:        ${analysis['base_value']}")
    print(f"  Efficiency bonus:  ${analysis['efficiency_bonus']}")
    print(f"  Total value:       ${analysis['total_value']}")
    print(f"\n  Value Breakdown:")
    for item in analysis['breakdown']:
        print(f"    {item['test_id']}: ${item['value']:4d} ({item['type']}) - {item['reason']}")

Business Value Analysis

Llama 3.3 70B:
  Base value:        $170
  Efficiency bonus:  $50
  Total value:       $220

  Value Breakdown:
    TEST_01: $  10 (business_lookup) - Passed all criteria
    TEST_02: $  50 (competitor_comparison) - Passed all criteria
    TEST_03: $ 100 (theme_retrieval) - Passed all criteria
    TEST_04: $   5 (proper_rejection) - Passed all criteria
    TEST_05: $   5 (proper_rejection) - Passed all criteria

GPT OSS 20B:
  Base value:        $15
  Efficiency bonus:  $50
  Total value:       $65

  Value Breakdown:
    TEST_01: $  10 (business_lookup) - Passed all criteria
    TEST_02: $  50 (competitor_comparison) - Passed all criteria
    TEST_03: $ -20 (theme_retrieval) - Failed quality criteria
    TEST_04: $   5 (proper_rejection) - Passed all criteria
    TEST_05: $ -30 (proper_rejection) - Failed to reject out-of-scope


### 11.5. Calculate Final ROI

Compute Return on Investment for each model and determine the winner.

In [0]:
# Calculate ROI for each model
roi_analysis = {}

for model in cost_analysis.keys():
    total_value = value_analysis[model]["total_value"]
    total_cost = cost_analysis[model]["total_cost"]
    
    net_benefit = total_value - total_cost
    roi_percentage = (net_benefit / total_cost) * 100 if total_cost > 0 else 0
    
    # Cost per correct answer
    passed_tests = quality_summary[model]["passed_tests"]
    cost_per_correct = total_cost / passed_tests if passed_tests > 0 else float('inf')
    
    # Value per dollar spent
    value_per_dollar = total_value / total_cost if total_cost > 0 else 0
    
    roi_analysis[model] = {
        "total_value": total_value,
        "total_cost": total_cost,
        "net_benefit": net_benefit,
        "roi_percentage": roi_percentage,
        "cost_per_correct": cost_per_correct,
        "value_per_dollar": value_per_dollar,
        "pass_rate": quality_summary[model]["pass_rate"]
    }

print("="*60)
print("ROI ANALYSIS RESULTS")
print("="*60)

for model, metrics in roi_analysis.items():
    print(f"\n{model}:")
    print(f"  Total Value Generated:    ${metrics['total_value']:.2f}")
    print(f"  Total Cost:               ${metrics['total_cost']:.6f}")
    print(f"  Net Benefit:              ${metrics['net_benefit']:.2f}")
    print(f"  ROI:                      {metrics['roi_percentage']:,.1f}%")
    print(f"  Pass Rate:                {metrics['pass_rate']:.1f}%")
    print(f"  Cost per Correct Answer:  ${metrics['cost_per_correct']:.6f}")
    print(f"  Value per Dollar Spent:   ${metrics['value_per_dollar']:,.2f}")

# Determine winner
print(f"\n{'='*60}")
print("WINNER DETERMINATION")
print("="*60)

models = list(roi_analysis.keys())
model_a = models[0]
model_b = models[1]

roi_diff = roi_analysis[model_a]["roi_percentage"] - roi_analysis[model_b]["roi_percentage"]
value_diff = roi_analysis[model_a]["total_value"] - roi_analysis[model_b]["total_value"]
cost_diff = roi_analysis[model_a]["total_cost"] - roi_analysis[model_b]["total_cost"]
quality_diff = roi_analysis[model_a]["pass_rate"] - roi_analysis[model_b]["pass_rate"]

if roi_analysis[model_a]["roi_percentage"] > roi_analysis[model_b]["roi_percentage"]:
    winner = model_a
    winner_roi = roi_analysis[model_a]["roi_percentage"]
else:
    winner = model_b
    winner_roi = roi_analysis[model_b]["roi_percentage"]

print(f"\nWinner: {winner}")
print(f"  ROI: {winner_roi:,.1f}%")
print(f"\nDifferences ({model_a} vs {model_b}):")
print(f"  ROI:          {roi_diff:+,.1f} percentage points")
print(f"  Total Value:  ${value_diff:+.2f}")
print(f"  Total Cost:   ${cost_diff:+.6f}")
print(f"  Pass Rate:    {quality_diff:+.1f} percentage points")

ROI ANALYSIS RESULTS

Llama 3.3 70B:
  Total Value Generated:    $220.00
  Total Cost:               $0.003567
  Net Benefit:              $220.00
  ROI:                      6,166,683.5%
  Pass Rate:                100.0%
  Cost per Correct Answer:  $0.000713
  Value per Dollar Spent:   $61,667.83

GPT OSS 20B:
  Total Value Generated:    $65.00
  Total Cost:               $0.000545
  Net Benefit:              $65.00
  ROI:                      11,930,883.8%
  Pass Rate:                60.0%
  Cost per Correct Answer:  $0.000182
  Value per Dollar Spent:   $119,309.84

WINNER DETERMINATION

Winner: GPT OSS 20B
  ROI: 11,930,883.8%

Differences (Llama 3.3 70B vs GPT OSS 20B):
  ROI:          -5,764,200.4 percentage points
  Total Value:  $+155.00
  Total Cost:   $+0.003023
  Pass Rate:    +40.0 percentage points


### 11.6. Generate Comparison Tables

Create structured comparison tables for presentation.

In [0]:
# Create cost comparison table
import pandas as pd

cost_comparison = []
for model in cost_analysis.keys():
    cost_comparison.append({
        "Model": model,
        "Avg Tokens/Query": f"{token_summary[model]['avg_tokens_per_query']:.0f}",
        "Cost per Query": f"${cost_analysis[model]['cost_per_query']:.6f}",
        "Monthly Cost (1K queries)": f"${cost_analysis[model]['monthly_cost_1k_queries']:.2f}",
        "Total Eval Cost": f"${cost_analysis[model]['total_cost']:.6f}"
    })

cost_df = pd.DataFrame(cost_comparison)

print("="*60)
print("TABLE 1: COST COMPARISON")
print("="*60)
print()
display(cost_df)

TABLE 1: COST COMPARISON



Model,Avg Tokens/Query,Cost per Query,Monthly Cost (1K queries),Total Eval Cost
Llama 3.3 70B,1061,$0.000713,$0.71,$0.003567
GPT OSS 20B,1021,$0.000109,$0.11,$0.000545


In [0]:
# Create quality comparison table
quality_comparison = []
for model in quality_summary.keys():
    quality_comparison.append({
        "Model": model,
        "LLM Judge Pass Rate": f"{quality_summary[model]['pass_rate']:.1f}%",
        "Groundedness": f"{quality_summary[model]['dimension_rates']['groundedness']:.1f}%",
        "Relevance": f"{quality_summary[model]['dimension_rates']['relevance']:.1f}%",
        "Usefulness": f"{quality_summary[model]['dimension_rates']['usefulness']:.1f}%",
        "Tool Accuracy": f"{quality_summary[model]['dimension_rates']['tool_use']:.1f}%",
        "Rejection Accuracy": f"{quality_summary[model]['dimension_rates']['scope']:.1f}%"
    })

quality_df = pd.DataFrame(quality_comparison)

print("="*60)
print("TABLE 2: QUALITY COMPARISON")
print("="*60)
print()
display(quality_df)

TABLE 2: QUALITY COMPARISON



Model,LLM Judge Pass Rate,Groundedness,Relevance,Usefulness,Tool Accuracy,Rejection Accuracy
Llama 3.3 70B,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%
GPT OSS 20B,60.0%,100.0%,80.0%,60.0%,100.0%,80.0%


In [0]:
# Create ROI summary table
roi_comparison = []
for model in roi_analysis.keys():
    roi_comparison.append({
        "Model": model,
        "Total Value": f"${roi_analysis[model]['total_value']:.2f}",
        "Total Cost": f"${roi_analysis[model]['total_cost']:.6f}",
        "Net Benefit": f"${roi_analysis[model]['net_benefit']:.2f}",
        "ROI %": f"{roi_analysis[model]['roi_percentage']:,.1f}%",
        "Cost per Correct": f"${roi_analysis[model]['cost_per_correct']:.6f}",
        "Value/$": f"${roi_analysis[model]['value_per_dollar']:,.2f}"
    })

roi_df = pd.DataFrame(roi_comparison)

print("="*60)
print("TABLE 3: ROI SUMMARY")
print("="*60)
print()
display(roi_df)

TABLE 3: ROI SUMMARY



Model,Total Value,Total Cost,Net Benefit,ROI %,Cost per Correct,Value/$
Llama 3.3 70B,$220.00,$0.003567,$220.00,"6,166,683.5%",$0.000713,"$61,667.83"
GPT OSS 20B,$65.00,$0.000545,$65.00,"11,930,883.8%",$0.000182,"$119,309.84"


### 11.7. Sensitivity Analysis

Test ROI under different scenarios to understand robustness of the recommendation.

In [0]:
# Sensitivity analysis: different volume scenarios
scenarios = {
    "Low Volume (100 queries/month)": 100,
    "Medium Volume (1,000 queries/month)": 1000,
    "High Volume (10,000 queries/month)": 10000,
    "Enterprise (100,000 queries/month)": 100000
}

print("="*60)
print("SENSITIVITY ANALYSIS: Volume Scenarios")
print("="*60)

for scenario_name, monthly_queries in scenarios.items():
    print(f"\n{scenario_name}:")
    print(f"  {'Model':<20s} {'Monthly Cost':<15s} {'Monthly Value':<15s} {'Net Benefit':<15s} {'ROI':<15s}")
    print(f"  {'-'*80}")
    
    for model in cost_analysis.keys():
        monthly_cost = cost_analysis[model]["cost_per_query"] * monthly_queries
        
        # Assume pass rate stays constant
        queries_passed = monthly_queries * (quality_summary[model]["pass_rate"] / 100)
        
        # Calculate average value per query
        avg_value_per_query = value_analysis[model]["total_value"] / 5  # 5 test queries
        monthly_value = avg_value_per_query * monthly_queries
        
        net_benefit = monthly_value - monthly_cost
        roi = (net_benefit / monthly_cost) * 100 if monthly_cost > 0 else 0
        
        print(f"  {model:<20s} ${monthly_cost:<14.2f} ${monthly_value:<14.2f} ${net_benefit:<14.2f} {roi:>13,.0f}%")

SENSITIVITY ANALYSIS: Volume Scenarios

Low Volume (100 queries/month):
  Model                Monthly Cost    Monthly Value   Net Benefit     ROI            
  --------------------------------------------------------------------------------
  Llama 3.3 70B        $0.07           $4400.00        $4399.93            6,166,683%
  GPT OSS 20B          $0.01           $1300.00        $1299.99           11,930,884%

Medium Volume (1,000 queries/month):
  Model                Monthly Cost    Monthly Value   Net Benefit     ROI            
  --------------------------------------------------------------------------------
  Llama 3.3 70B        $0.71           $44000.00       $43999.29           6,166,683%
  GPT OSS 20B          $0.11           $13000.00       $12999.89          11,930,884%

High Volume (10,000 queries/month):
  Model                Monthly Cost    Monthly Value   Net Benefit     ROI            
  --------------------------------------------------------------------------------

In [0]:
# Sensitivity analysis: quality threshold requirements
print("\n" + "="*60)
print("SENSITIVITY ANALYSIS: Quality Thresholds")
print("="*60)

quality_thresholds = [70, 80, 90, 95, 100]

print(f"\n{'Threshold':<12s} {'Llama 3.3 70B':<20s} {'GPT OSS 20B':<20s}")
print(f"{'-'*52}")

for threshold in quality_thresholds:
    llama_meets = "✓ Qualifies" if quality_summary["Llama 3.3 70B"]["pass_rate"] >= threshold else "✗ Does not qualify"
    gpt_meets = "✓ Qualifies" if quality_summary["GPT OSS 20B"]["pass_rate"] >= threshold else "✗ Does not qualify"
    
    print(f"{threshold}% pass rate: {llama_meets:<20s} {gpt_meets:<20s}")

print(f"\nConclusion:")
print(f"  For quality-critical applications requiring >90% pass rate,")
if quality_summary["Llama 3.3 70B"]["pass_rate"] >= 90:
    print(f"  Llama 3.3 70B is the only viable option.")
elif quality_summary["GPT OSS 20B"]["pass_rate"] >= 90:
    print(f"  GPT OSS 20B is the only viable option.")
else:
    print(f"  Neither model currently meets the threshold.")
    print(f"  Consider prompt engineering or model fine-tuning.")


SENSITIVITY ANALYSIS: Quality Thresholds

Threshold    Llama 3.3 70B        GPT OSS 20B         
----------------------------------------------------
70% pass rate: ✓ Qualifies          ✗ Does not qualify  
80% pass rate: ✓ Qualifies          ✗ Does not qualify  
90% pass rate: ✓ Qualifies          ✗ Does not qualify  
95% pass rate: ✓ Qualifies          ✗ Does not qualify  
100% pass rate: ✓ Qualifies          ✗ Does not qualify  

Conclusion:
  For quality-critical applications requiring >90% pass rate,
  Llama 3.3 70B is the only viable option.


### 11.8. Final Recommendation

Based on the comprehensive ROI analysis, here is the recommended model selection strategy.

In [0]:
# Generate final recommendation
print("="*60)
print("FINAL RECOMMENDATION")
print("="*60)

print(f"\n🏆 Recommended Model: {winner}\n")

print("Rationale:")
if winner == "Llama 3.3 70B":
    print(f"  ✓ Higher quality: {quality_summary[winner]['pass_rate']:.1f}% pass rate")
    print(f"  ✓ Superior ROI: {roi_analysis[winner]['roi_percentage']:,.0f}%")
    print(f"  ✓ Better reliability: Passed all rejection tests")
    print(f"  ✓ Strong performance across all dimensions")
    print(f"  ✓ Value per dollar: ${roi_analysis[winner]['value_per_dollar']:,.2f}")
else:
    print(f"  ✓ Lower cost: ${cost_analysis[winner]['cost_per_query']:.6f} per query")
    print(f"  ✓ Acceptable quality: {quality_summary[winner]['pass_rate']:.1f}% pass rate")
    print(f"  ✓ Better cost efficiency for high-volume scenarios")
    print(f"  ✓ Suitable for budget-constrained deployments")

print("\nWhen to use Llama 3.3 70B:")
print("  • Quality-critical applications (>80% pass rate required)")
print("  • Complex competitor comparisons requiring nuanced analysis")
print("  • Strategic decision support for restaurant owners")
print("  • When accuracy is more important than cost")

print("\nWhen to use GPT OSS 20B:")
print("  • High-volume, cost-sensitive applications")
print("  • Simple lookup queries with lower stakes")
print("  • Budget-constrained environments")
print("  • When acceptable quality at lowest cost is the priority")

print(f"\nExpected ROI: {roi_analysis[winner]['roi_percentage']:,.0f}%")
print(f"Monthly cost at 1K queries: ${cost_analysis[winner]['monthly_cost_1k_queries']:.2f}")
print(f"Pass rate: {quality_summary[winner]['pass_rate']:.1f}%")

print("\n" + "="*60)
print("END OF ROI ANALYSIS")
print("="*60)

FINAL RECOMMENDATION

🏆 Recommended Model: GPT OSS 20B

Rationale:
  ✓ Lower cost: $0.000109 per query
  ✓ Acceptable quality: 60.0% pass rate
  ✓ Better cost efficiency for high-volume scenarios
  ✓ Suitable for budget-constrained deployments

When to use Llama 3.3 70B:
  • Quality-critical applications (>80% pass rate required)
  • Complex competitor comparisons requiring nuanced analysis
  • Strategic decision support for restaurant owners
  • When accuracy is more important than cost

When to use GPT OSS 20B:
  • High-volume, cost-sensitive applications
  • Simple lookup queries with lower stakes
  • Budget-constrained environments
  • When acceptable quality at lowest cost is the priority

Expected ROI: 11,930,884%
Monthly cost at 1K queries: $0.11
Pass rate: 60.0%

END OF ROI ANALYSIS


In [0]:
# Save comparison tables to Unity Catalog for easy access in presentation
# Sanitize column names for Delta Lake compatibility (remove spaces, special characters)

def sanitize_column_names(df):
    """Replace invalid characters in column names with underscores"""
    for col_name in df.columns:
        new_col_name = col_name.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "").replace("%", "pct").replace("$", "dollars")
        df = df.withColumnRenamed(col_name, new_col_name)
    return df

# Cost comparison
cost_comparison_df = spark.createDataFrame(pd.DataFrame(cost_comparison))
cost_comparison_df = sanitize_column_names(cost_comparison_df)
cost_comparison_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.roi_cost_comparison")

# Quality comparison
quality_comparison_df = spark.createDataFrame(pd.DataFrame(quality_comparison))
quality_comparison_df = sanitize_column_names(quality_comparison_df)
quality_comparison_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.roi_quality_comparison")

# ROI summary
roi_summary_df = spark.createDataFrame(pd.DataFrame(roi_comparison))
roi_summary_df = sanitize_column_names(roi_summary_df)
roi_summary_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.roi_summary")

print("="*60)
print("ROI Analysis Tables Saved")
print("="*60)
print(f"\nTables created in {CATALOG}.{SCHEMA}:")
print(f"  • roi_cost_comparison")
print(f"  • roi_quality_comparison")
print(f"  • roi_summary")
print("\nThese tables can be used for:")
print("  • Creating dashboard visualizations")
print("  • Generating presentation slides")
print("  • Sharing with stakeholders")
print("  • Further analysis and reporting")

ROI Analysis Tables Saved

Tables created in main.aai510_final_agent:
  • roi_cost_comparison
  • roi_quality_comparison
  • roi_summary

These tables can be used for:
  • Creating dashboard visualizations
  • Generating presentation slides
  • Sharing with stakeholders
  • Further analysis and reporting
